In [ ]:
# Runtime -> Change runtime type -> Hardware accelerator -> GPU
!pip -q install pandas numpy scikit-learn torch joblib pyarrow


In [ ]:
from pathlib import Path

DB_PATH = Path('/content/stock_prices_20y.db')
OUTPUT_DIR = Path('/content/model_outputs')
MODELS_DIR = OUTPUT_DIR / 'models'
RESULTS_DIR = OUTPUT_DIR / 'results'
CONFIGS_DIR = OUTPUT_DIR / 'configs'
CACHE_DIR = OUTPUT_DIR / 'rnn_cache'

for path in [MODELS_DIR, RESULTS_DIR, CONFIGS_DIR, CACHE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(f'Missing SQLite database: {DB_PATH}')

print('DB_PATH    =', DB_PATH)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('CACHE_DIR  =', CACHE_DIR)


In [ ]:
import gc
import json
import os
import pickle
import random
import sqlite3
import time
from dataclasses import asdict, dataclass
from datetime import datetime
from typing import List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, Dataset

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU_ENABLED = DEVICE.type == 'cuda'
AMP_ENABLED = GPU_ENABLED
NUM_WORKERS = 1
RUN_TAG = datetime.utcnow().strftime('%Y%m%d_%H%M%S')

print('Torch device:', DEVICE)
if GPU_ENABLED:
    print('GPU:', torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
else:
    print('No GPU detected. The notebook still runs, just slower.')

with sqlite3.connect(DB_PATH) as conn:
    row_count = conn.execute('SELECT COUNT(*) FROM prices').fetchone()[0]
    ticker_count = conn.execute('SELECT COUNT(DISTINCT ticker) FROM prices').fetchone()[0]
    min_date, max_date = conn.execute('SELECT MIN(date), MAX(date) FROM prices').fetchone()

print(f'Rows: {row_count:,} | Tickers: {ticker_count:,} | Date range: {min_date} -> {max_date}')

@dataclass
class Config:
    db_path: str = str(DB_PATH)
    table_name: str = 'prices'
    n_splits: int = 3
    test_size: float = 0.2
    target_horizon: int = 1
    predict_target: str = 'return'
    random_state: int = 42
    sequence_length: int = 30
    batch_size: int = 1024 if GPU_ENABLED else 512
    epochs: int = 35
    patience: int = 8
    rnn_units: int = 96
    num_layers: int = 2
    dense_units: int = 48
    dropout: float = 0.25
    learning_rate: float = 7e-4
    weight_decay: float = 1e-4
    max_tickers: int = 0
    rebuild_cache: bool = True
    cache_dir: str = str(CACHE_DIR)
    model_output_path: str = str(MODELS_DIR / f'{RUN_TAG}-rnn_torch.pkl')
    config_output_path: str = str(CONFIGS_DIR / f'{RUN_TAG}-rnn_torch_config.json')
    feature_cols: tuple = (
        'log_ret_1', 'log_ret_3', 'log_ret_5', 'log_ret_10', 'log_ret_20',
        'hl_range', 'oc_change', 'close_vs_ma_5', 'close_vs_ma_10',
        'close_vs_ma_20', 'close_vs_ma_50', 'ma_5_vs_ma_20',
        'rolling_vol_5', 'rolling_vol_20', 'log_vol_chg_1',
        'vol_vs_ma_5', 'vol_vs_ma_20', 'rsi_14', 'macd_norm',
        'bb_pos', 'atr_14_norm',
    )

cfg = Config()
print(cfg)


def set_seed(seed: int) -> None:
    np.random.seed(seed)
    torch.manual_seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_tickers(conn: sqlite3.Connection) -> list[str]:
    return [r[0] for r in conn.execute(f'SELECT DISTINCT ticker FROM {cfg.table_name} ORDER BY ticker')]


def load_ticker_prices(conn: sqlite3.Connection, ticker: str) -> pd.DataFrame:
    df = pd.read_sql_query(
        f'''SELECT ticker, date, open, high, low, close, adj_close, volume
            FROM {cfg.table_name}
            WHERE ticker = ?
            ORDER BY date''',
        conn,
        params=[ticker],
    )
    df['date'] = pd.to_datetime(df['date'])
    for c in ['open', 'high', 'low', 'close', 'adj_close']:
        df[c] = pd.to_numeric(df[c], errors='coerce').astype('float32')
    df['volume'] = pd.to_numeric(df['volume'], errors='coerce').fillna(0).astype('float32')
    return df


def make_sequence_features_for_ticker(df: pd.DataFrame) -> pd.DataFrame:
    g = df.sort_values('date').copy()
    close = g['close'].astype('float64')
    high = g['high'].astype('float64')
    low = g['low'].astype('float64')
    open_ = g['open'].astype('float64')
    volume = g['volume'].astype('float64').clip(lower=1.0)
    log_close = np.log(close)
    g['log_ret_1'] = log_close.diff(1)
    g['log_ret_3'] = log_close.diff(3)
    g['log_ret_5'] = log_close.diff(5)
    g['log_ret_10'] = log_close.diff(10)
    g['log_ret_20'] = log_close.diff(20)
    g['hl_range'] = (high - low) / close
    g['oc_change'] = (close - open_) / open_.clip(lower=1e-6)
    ma5 = close.rolling(5).mean()
    ma10 = close.rolling(10).mean()
    ma20 = close.rolling(20).mean()
    ma50 = close.rolling(50).mean()
    g['close_vs_ma_5'] = close / ma5.clip(lower=1e-6) - 1.0
    g['close_vs_ma_10'] = close / ma10.clip(lower=1e-6) - 1.0
    g['close_vs_ma_20'] = close / ma20.clip(lower=1e-6) - 1.0
    g['close_vs_ma_50'] = close / ma50.clip(lower=1e-6) - 1.0
    g['ma_5_vs_ma_20'] = ma5 / ma20.clip(lower=1e-6) - 1.0
    g['rolling_vol_5'] = g['log_ret_1'].rolling(5).std()
    g['rolling_vol_20'] = g['log_ret_1'].rolling(20).std()
    log_vol = np.log1p(volume)
    g['log_vol_chg_1'] = log_vol.diff(1)
    vol_ma5 = volume.rolling(5).mean()
    vol_ma20 = volume.rolling(20).mean()
    g['vol_vs_ma_5'] = volume / vol_ma5.clip(lower=1.0) - 1.0
    g['vol_vs_ma_20'] = volume / vol_ma20.clip(lower=1.0) - 1.0
    delta = close.diff()
    gain = delta.clip(lower=0).ewm(alpha=1.0 / 14, min_periods=14).mean()
    loss = (-delta.clip(upper=0)).ewm(alpha=1.0 / 14, min_periods=14).mean()
    g['rsi_14'] = (100.0 - 100.0 / (1.0 + gain / (loss + 1e-8))) / 100.0 - 0.5
    ema12 = close.ewm(span=12, min_periods=12).mean()
    ema26 = close.ewm(span=26, min_periods=26).mean()
    g['macd_norm'] = (ema12 - ema26) / close.clip(lower=1e-6)
    bb_mean = close.rolling(20).mean()
    bb_std = close.rolling(20).std()
    bb_upper = bb_mean + 2 * bb_std
    bb_lower = bb_mean - 2 * bb_std
    g['bb_pos'] = (close - bb_lower) / (bb_upper - bb_lower).clip(lower=1e-6) - 0.5
    prev_close = close.shift(1)
    tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
    g['atr_14_norm'] = tr.ewm(alpha=1.0 / 14, min_periods=14).mean() / close.clip(lower=1e-6)
    return g.replace([np.inf, -np.inf], np.nan)


def split_train_test_single_ticker(df: pd.DataFrame, test_size: float, embargo: int):
    g = df.sort_values('date').reset_index(drop=True)
    n = len(g)
    split_idx = int(n * (1 - test_size))
    train_end = split_idx - embargo
    if split_idx <= 0 or split_idx >= n or train_end <= 0:
        return None, None
    return g.iloc[:train_end].copy(), g.iloc[split_idx:].copy()


def add_target_single_ticker(df: pd.DataFrame, horizon: int, predict_target: str) -> pd.DataFrame:
    g = df.sort_values('date').copy()
    close = g['close']
    if predict_target == 'return':
        g['target'] = np.log(close.shift(-horizon) / close).astype('float32')
    elif predict_target == 'price':
        g['target'] = close.shift(-horizon).astype('float32')
    else:
        raise ValueError("predict_target must be 'return' or 'price'")
    return g


def build_sequences_for_ticker(df: pd.DataFrame, feature_cols: list[str], sequence_length: int):
    if df is None or df.empty or len(df) < sequence_length:
        return None, None, None
    g = df.sort_values('date').reset_index(drop=True)
    feat = g[feature_cols].to_numpy(dtype=np.float32)
    target = g['target'].to_numpy(dtype=np.float32)
    dates = g['date'].to_numpy('datetime64[ns]')
    closes = g['close'].to_numpy(dtype=np.float32)
    xs, ys, metas = [], [], []
    for i in range(sequence_length - 1, len(g)):
        if np.isnan(target[i]):
            continue
        x_seq = feat[i - sequence_length + 1 : i + 1]
        if np.isnan(x_seq).any():
            continue
        xs.append(x_seq)
        ys.append(target[i])
        metas.append((dates[i], closes[i]))
    if not xs:
        return None, None, None
    return np.asarray(xs, dtype=np.float32), np.asarray(ys, dtype=np.float32), metas


def count_sequences_for_ticker(df: pd.DataFrame, feature_cols: list[str], sequence_length: int) -> int:
    X, _, _ = build_sequences_for_ticker(df, feature_cols, sequence_length)
    return 0 if X is None else int(len(X))


def build_grouped_time_folds(meta: pd.DataFrame, n_splits: int, embargo: int) -> List[Tuple[np.ndarray, np.ndarray]]:
    meta = meta.sort_values(['ticker', 'date']).reset_index(drop=True)
    folds = []
    for fold_num in range(n_splits):
        tr_idx_all, va_idx_all = [], []
        for _, grp in meta.groupby('ticker', sort=False):
            idx = grp.index.to_numpy()
            n = len(idx)
            if n < (n_splits + 1) * max(embargo, 1) + 5:
                continue
            boundaries = np.linspace(0, n, n_splits + 2, dtype=int)
            va_start = boundaries[fold_num + 1]
            va_end = boundaries[fold_num + 2]
            purged_train_end = va_start - embargo
            if purged_train_end <= 0 or va_end <= va_start:
                continue
            tr_idx_all.extend(idx[:purged_train_end].tolist())
            va_idx_all.extend(idx[va_start:va_end].tolist())
        if tr_idx_all and va_idx_all:
            folds.append((np.asarray(tr_idx_all, dtype=np.int64), np.asarray(va_idx_all, dtype=np.int64)))
    if not folds:
        raise ValueError('No CV folds built.')
    return folds


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) < 2:
        return {'rmse': np.nan, 'mae': np.nan, 'r2': np.nan, 'directional_accuracy': np.nan}
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae': float(mean_absolute_error(y_true, y_pred)),
        'r2': float(r2_score(y_true, y_pred)),
        'directional_accuracy': float(np.mean(np.sign(y_true) == np.sign(y_pred))),
    }


class MemmapSequenceDataset(Dataset):
    def __init__(self, x_path, y_path, total_len, indices, scaler, sequence_length, n_features):
        self.x_path = x_path
        self.y_path = y_path
        self.total_len = total_len
        self.indices = np.asarray(indices, dtype=np.int64)
        self.sequence_length = sequence_length
        self.n_features = n_features
        self.mean = scaler.mean_.astype(np.float32)
        self.scale = scaler.scale_.astype(np.float32)
        self.scale[self.scale == 0.0] = 1.0
        self.X = None
        self.y = None

    def _ensure_open(self):
        if self.X is None:
            self.X = np.memmap(self.x_path, dtype=np.float16, mode='r', shape=(self.total_len, self.sequence_length, self.n_features))
            self.y = np.memmap(self.y_path, dtype=np.float32, mode='r', shape=(self.total_len,))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        self._ensure_open()
        i = int(self.indices[idx])
        x = np.asarray(self.X[i], dtype=np.float32)
        x = (x - self.mean) / self.scale
        return torch.from_numpy(x), torch.tensor(float(self.y[i]), dtype=torch.float32)


class RNNModel(nn.Module):
    def __init__(self, n_features: int, cfg: Config):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=n_features,
            hidden_size=cfg.rnn_units,
            num_layers=cfg.num_layers,
            batch_first=True,
            nonlinearity='tanh',
            dropout=cfg.dropout if cfg.num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(cfg.dropout)
        self.fc1 = nn.Linear(cfg.rnn_units, cfg.dense_units)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(cfg.dense_units, 1)

    def forward(self, x):
        rnn_out, _ = self.rnn(x)
        last = rnn_out[:, -1, :]
        out = self.dropout(last)
        out = self.relu(self.fc1(out))
        return self.fc2(out).squeeze(-1)


def make_loader(dataset, batch_size, shuffle):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=AMP_ENABLED, persistent_workers=False)


def train_model_amp(model, train_loader, val_loader, cfg):
    criterion = nn.HuberLoss(delta=1.0)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.learning_rate * 0.1)
    scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)
    best_val_loss = float('inf')
    best_state = None
    patience_counter = 0
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        total_loss, n_train = 0.0, 0
        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=AMP_ENABLED)
            yb = yb.to(DEVICE, non_blocking=AMP_ENABLED)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                pred = model(xb)
                loss = criterion(pred, yb)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            total_loss += loss.item() * len(yb)
            n_train += len(yb)
        scheduler.step()
        train_loss = total_loss / max(n_train, 1)
        model.eval()
        total_val, n_val = 0.0, 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE, non_blocking=AMP_ENABLED)
                yb = yb.to(DEVICE, non_blocking=AMP_ENABLED)
                with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
                    pred = model(xb)
                    val_loss = criterion(pred, yb)
                total_val += val_loss.item() * len(yb)
                n_val += len(yb)
        val_loss = total_val / max(n_val, 1)
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
        print(f'Epoch {epoch:3d} | train={train_loss:.6f} | val={val_loss:.6f} | lr={scheduler.get_last_lr()[0]:.2e}')
        if patience_counter >= cfg.patience:
            print(f'Early stop at epoch {epoch}')
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model


@torch.no_grad()
def predict_model(model, loader):
    model.eval()
    preds = []
    for xb, _ in loader:
        xb = xb.to(DEVICE, non_blocking=AMP_ENABLED)
        with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=AMP_ENABLED):
            preds.append(model(xb).detach().cpu().numpy())
    return np.concatenate(preds)


def estimate_and_cache_sequences(cfg: Config):
    feature_cols = list(cfg.feature_cols)
    cache_dir = Path(cfg.cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    x_train_path = cache_dir / 'train_X.dat'
    y_train_path = cache_dir / 'train_y.dat'
    x_test_path = cache_dir / 'test_X.dat'
    y_test_path = cache_dir / 'test_y.dat'
    meta_train_path = cache_dir / 'meta_train.parquet'
    meta_test_path = cache_dir / 'meta_test.parquet'
    scaler_path = cache_dir / 'scaler.pkl'
    if (not cfg.rebuild_cache and all(p.exists() for p in [x_train_path, y_train_path, x_test_path, y_test_path, meta_train_path, meta_test_path, scaler_path])):
        print('Using existing cache from', cache_dir)
        meta_train = pd.read_parquet(meta_train_path)
        meta_test = pd.read_parquet(meta_test_path)
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
        return {'x_train_path': str(x_train_path), 'y_train_path': str(y_train_path), 'x_test_path': str(x_test_path), 'y_test_path': str(y_test_path), 'meta_train': meta_train, 'meta_test': meta_test, 'scaler': scaler, 'n_train': len(meta_train), 'n_test': len(meta_test)}

    with sqlite3.connect(cfg.db_path) as conn:
        tickers = get_tickers(conn)
    if cfg.max_tickers > 0 and len(tickers) > cfg.max_tickers:
        rng = np.random.default_rng(cfg.random_state)
        tickers = rng.choice(tickers, cfg.max_tickers, replace=False).tolist()
    print(f'Caching sequences for {len(tickers)} tickers...')

    train_count = 0
    test_count = 0
    with sqlite3.connect(cfg.db_path) as conn:
        for idx, ticker in enumerate(tickers, start=1):
            raw = load_ticker_prices(conn, ticker)
            feat = make_sequence_features_for_ticker(raw)
            train_feat, test_feat = split_train_test_single_ticker(feat, cfg.test_size, cfg.target_horizon)
            if train_feat is None:
                continue
            train_df = add_target_single_ticker(train_feat, cfg.target_horizon, cfg.predict_target)
            test_df = add_target_single_ticker(test_feat, cfg.target_horizon, cfg.predict_target)
            train_count += count_sequences_for_ticker(train_df, feature_cols, cfg.sequence_length)
            test_count += count_sequences_for_ticker(test_df, feature_cols, cfg.sequence_length)
            if idx % 50 == 0:
                print(f'Count pass: {idx}/{len(tickers)} tickers | train={train_count:,} | test={test_count:,}')

    print(f'Allocating memmaps | train={train_count:,} | test={test_count:,}')
    n_features = len(feature_cols)
    X_train = np.memmap(x_train_path, dtype=np.float16, mode='w+', shape=(train_count, cfg.sequence_length, n_features))
    y_train = np.memmap(y_train_path, dtype=np.float32, mode='w+', shape=(train_count,))
    X_test = np.memmap(x_test_path, dtype=np.float16, mode='w+', shape=(test_count, cfg.sequence_length, n_features))
    y_test = np.memmap(y_test_path, dtype=np.float32, mode='w+', shape=(test_count,))
    scaler = StandardScaler()
    meta_train_rows = []
    meta_test_rows = []
    train_pos = 0
    test_pos = 0

    with sqlite3.connect(cfg.db_path) as conn:
        for idx, ticker in enumerate(tickers, start=1):
            raw = load_ticker_prices(conn, ticker)
            feat = make_sequence_features_for_ticker(raw)
            train_feat, test_feat = split_train_test_single_ticker(feat, cfg.test_size, cfg.target_horizon)
            if train_feat is None:
                continue
            train_df = add_target_single_ticker(train_feat, cfg.target_horizon, cfg.predict_target)
            test_df = add_target_single_ticker(test_feat, cfg.target_horizon, cfg.predict_target)
            X_tr, y_tr, meta_tr = build_sequences_for_ticker(train_df, feature_cols, cfg.sequence_length)
            if X_tr is not None:
                n = len(X_tr)
                X_train[train_pos:train_pos+n] = X_tr.astype(np.float16)
                y_train[train_pos:train_pos+n] = y_tr
                scaler.partial_fit(X_tr.reshape(-1, n_features))
                meta_train_rows.extend({'ticker': ticker, 'date': pd.Timestamp(d), 'close': float(c)} for d, c in meta_tr)
                train_pos += n
            X_te, y_te, meta_te = build_sequences_for_ticker(test_df, feature_cols, cfg.sequence_length)
            if X_te is not None:
                n = len(X_te)
                X_test[test_pos:test_pos+n] = X_te.astype(np.float16)
                y_test[test_pos:test_pos+n] = y_te
                meta_test_rows.extend({'ticker': ticker, 'date': pd.Timestamp(d), 'close': float(c)} for d, c in meta_te)
                test_pos += n
            del raw, feat, train_feat, test_feat, train_df, test_df, X_tr, y_tr, meta_tr, X_te, y_te, meta_te
            gc.collect()
            if idx % 25 == 0:
                print(f'Write pass: {idx}/{len(tickers)} tickers | train={train_pos:,} | test={test_pos:,}')

    X_train.flush(); y_train.flush(); X_test.flush(); y_test.flush()
    meta_train = pd.DataFrame(meta_train_rows)
    meta_test = pd.DataFrame(meta_test_rows)
    meta_train.to_parquet(meta_train_path, index=False)
    meta_test.to_parquet(meta_test_path, index=False)
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print('Cache build finished.')
    return {'x_train_path': str(x_train_path), 'y_train_path': str(y_train_path), 'x_test_path': str(x_test_path), 'y_test_path': str(y_test_path), 'meta_train': meta_train, 'meta_test': meta_test, 'scaler': scaler, 'n_train': train_pos, 'n_test': test_pos}


In [ ]:
start_time = time.perf_counter()
set_seed(cfg.random_state)

cache = estimate_and_cache_sequences(cfg)
meta_train = cache['meta_train']
meta_test = cache['meta_test']
scaler = cache['scaler']
n_train = cache['n_train']
n_test = cache['n_test']
n_features = len(cfg.feature_cols)

print(f'Cached train sequences: {n_train:,}')
print(f'Cached test sequences : {n_test:,}')

y_train_all = np.memmap(cache['y_train_path'], dtype=np.float32, mode='r', shape=(n_train,))
y_test_all = np.memmap(cache['y_test_path'], dtype=np.float32, mode='r', shape=(n_test,))

folds = build_grouped_time_folds(meta_train, cfg.n_splits, cfg.target_horizon)
print('CV folds:', len(folds))

cv_results = []
for fold_i, (tr_idx, va_idx) in enumerate(folds, start=1):
    print(f'\n=== Fold {fold_i}/{len(folds)} ===')
    train_ds = MemmapSequenceDataset(cache['x_train_path'], cache['y_train_path'], n_train, tr_idx, scaler, cfg.sequence_length, n_features)
    val_ds = MemmapSequenceDataset(cache['x_train_path'], cache['y_train_path'], n_train, va_idx, scaler, cfg.sequence_length, n_features)
    train_loader = make_loader(train_ds, cfg.batch_size, True)
    val_loader = make_loader(val_ds, cfg.batch_size, False)
    model = RNNModel(n_features, cfg).to(DEVICE)
    model = train_model_amp(model, train_loader, val_loader, cfg)
    metrics = regression_metrics(np.asarray(y_train_all[va_idx]), predict_model(model, val_loader))
    cv_results.append(metrics)
    print(metrics)
    del model, train_ds, val_ds, train_loader, val_loader
    gc.collect()
    if GPU_ENABLED:
        torch.cuda.empty_cache()

cv_df = pd.DataFrame(cv_results)
print('\nCV mean metrics')
print(cv_df.mean(numeric_only=True).to_string())

all_idx = np.arange(n_train, dtype=np.int64)
split = int(n_train * 0.9)
train_ds = MemmapSequenceDataset(cache['x_train_path'], cache['y_train_path'], n_train, all_idx[:split], scaler, cfg.sequence_length, n_features)
val_ds = MemmapSequenceDataset(cache['x_train_path'], cache['y_train_path'], n_train, all_idx[split:], scaler, cfg.sequence_length, n_features)
train_loader = make_loader(train_ds, cfg.batch_size, True)
val_loader = make_loader(val_ds, cfg.batch_size, False)

final_model = RNNModel(n_features, cfg).to(DEVICE)
final_model = train_model_amp(final_model, train_loader, val_loader, cfg)

test_idx = np.arange(n_test, dtype=np.int64)
test_ds = MemmapSequenceDataset(cache['x_test_path'], cache['y_test_path'], n_test, test_idx, scaler, cfg.sequence_length, n_features)
test_loader = make_loader(test_ds, cfg.batch_size, False)
y_test_pred = predict_model(final_model, test_loader)
test_metrics = regression_metrics(np.asarray(y_test_all), y_test_pred)
print('Held-out test metrics:', test_metrics)

cpu_state_dict = {k: v.detach().cpu() for k, v in final_model.state_dict().items()}
payload = {'model_state_dict': cpu_state_dict, 'scaler': scaler, 'config': asdict(cfg), 'cv_results': cv_results, 'test_metrics': test_metrics}
with open(cfg.model_output_path, 'wb') as f:
    pickle.dump(payload, f)
config_dict = asdict(cfg)
config_dict['feature_cols'] = list(config_dict['feature_cols'])
with open(cfg.config_output_path, 'w') as f:
    json.dump(config_dict, f, indent=2)

pred_path = RESULTS_DIR / f'{RUN_TAG}-rnn_predictions.csv'
out = meta_test.copy()
out['y_true'] = np.asarray(y_test_all)
out['y_pred'] = y_test_pred
out.to_csv(pred_path, index=False)

elapsed_min = (time.perf_counter() - start_time) / 60
print(f'\nSaved model      : {cfg.model_output_path}')
print(f'Saved config     : {cfg.config_output_path}')
print(f'Saved predictions: {pred_path}')
print(f'Elapsed minutes  : {elapsed_min:.2f}')
